In [0]:
import data.utilities.common as Commonlib
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU

In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed - this populates the appropriate configurations in the "2. Configuration File" drop-down
CONFIG_BASE_PATH = "../../configuration/excel/"

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "2. Configuration File")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}{subject_area}/{config_file}"

config = Commonlib.get_object_config(config_file_path)
source_config = config.get("source")
bronze_config = config.get(ML.bronze)

In [0]:
try:
    medallion = Medallion(config)

    # check if job requires file extraction before proceeding
    medallion.extract_to_raw()

except NotImplementedError as nie:
    logger.warn(f"excel_procedure warning. {nie}")
except Exception as e:
    medallion.logger.error(
        f"excel_procedure failed during extraction. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
try:
    # get a list of all files in the raw directory
    raw_file_list = medallion.list_raw_files()
    medallion.logger.info(
        f"excel_procedure loading started. total_files_found={len(raw_file_list)}, subject={subject_area}, config_file={config_file}"
    )

    for file in raw_file_list:
        spark_excel_options = source_config.get("excel_options")
        medallion.read_raw_data(file.name, file_format="EXCEL", read_options=spark_excel_options)

        # write to bronze
        load_type = bronze_config.get("load_type", "append")
        try:
            medallion.write_to_delta(
                load_type=load_type,
                layer=ML.bronze,
                source=file.name,
            )
            medallion.df.unpersist()
        except Exception as e:
            medallion.logger.error(
                f"excel_procedure error while writing to bronze. subject={subject_area}, config_file={config_file}: {e}"
            )
            raise

        medallion.logger.info("excel_procedure file loading finished")
        # Move file to archive folder after processing
        if source_config.get("archive", True):
            medallion.archive(file.name)

    medallion.logger.info("excel_procedure finished")
except Exception as e:
    medallion.logger.error(
        f"excel_procedure failed during loading. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise